# AMML ranking 2026 — финальное решение

**Система:** Linux (Kaggle Notebook, GPU T4 x2, Python 3.13). Интернет включён (скачивание весов с HuggingFace).

Пайплайн: чистка (HTML, ударения, латинские буквы-двойники в русских словах) → лексические признаки и метаданные → 5 фолдов по md5(`query_id`) → zero-shot `deepvk/USER-bge-m3`, `BAAI/bge-reranker-v2-m3`, `jinaai/jina-reranker-v2-base-multilingual` и `Qwen/Qwen3-Reranker-8B` (без дообучения, признаки) → дообучение `bge-reranker-v2-m3` на train двумя способами: listwise по запросу (2 seed) и BCE по парам (1 seed), OOF по 5 фолдам → признаки относительно статьи и запроса → LightGBM lambdarank (3 seed) + CatBoost YetiRank → rank-average → `submission.csv`.

Дообучение — только на `train.csv` соревнования. Время работы ≈ 9 ч на T4 x2. Анализ и эксперименты — в README.md.

In [1]:
!pip install -q pymorphy3==2.0.6 pymorphy3-dicts-ru==2.4.417150.4580142 rank-bm25==0.2.2

In [2]:
import glob, os
src = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]) if os.path.exists('/kaggle/input') else '../input'
os.makedirs('data', exist_ok=True)
for f in ('train.csv', 'test.csv', 'sample_submission.csv'):
    if not os.path.exists(f'data/{f}'): os.symlink(f'{src}/{f}', f'data/{f}')
print(os.listdir('data'))

['test.csv', 'sample_submission.csv', 'train.csv']


## Код модулей

In [3]:
%%writefile feats.py
"""Cleaning + lexical features. Run: python feats.py -> data/feats.parquet"""
import html, re
import numpy as np, pandas as pd
import pymorphy3
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer

morph = pymorphy3.MorphAnalyzer()
_lem = {}
STOP = set('и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между сколько каком какие какое каких который которая которые году год'.split())


HOMO = str.maketrans('aceopxyABCEHKMOPTX', 'асеорхуАВСЕНКМОРТХ')  # Latin letters that look like Cyrillic ones


def fix_mixed(s):
    """In words that contain Cyrillic, Latin look-alikes are processing errors ('Cписок', 'ATФ'); real Latin words stay."""
    def fix(m):
        w = m[0].translate(HOMO)
        return w if re.search('[а-яё]', m[0], re.I) and not re.search('[a-z]', w, re.I) else m[0]
    return re.sub(r'\w+', fix, s)


def clean(s):
    if not isinstance(s, str):
        return ''
    s = html.unescape(html.unescape(s))
    s = re.sub(r'<[^>]+>', ' ', s)
    s = re.sub(r'^\s*Навигация:[^\n]*\n', '', s)
    s = s.replace('\xa0', ' ').replace(' ', ' ')
    s = s.replace('́', '')  # stress marks
    return fix_mixed(re.sub(r'\s+', ' ', s).strip())


def num(s):
    if not isinstance(s, str):
        return np.nan
    s = re.sub(r'[\s \xa0]', '', s).replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan


def lemmas(s):
    out = []
    for w in re.findall(r'[a-zа-яё0-9]+', s.lower()):
        if w not in _lem:
            _lem[w] = morph.parse(w)[0].normal_form
        l = _lem[w]
        if l not in STOP:
            out.append(l)
    return out


def build():
    tr = pd.read_csv('data/train.csv'); te = pd.read_csv('data/test.csv')
    df = pd.concat([tr, te], ignore_index=True)
    # stress marks (U+0301), taken before cleaning removes them: ~23% of queries, ~9% of passages (article lead paragraphs)
    raw = lambda c: df[c].fillna('').str.lower()
    q_stress, t_stress = raw('query').str.contains('\u0301'), (raw('title') + ' ' + raw('text')).str.contains('\u0301')
    sw = raw('query').str.findall(r'\w*\u0301\w*')
    stress_hit = [int(any(w in x for w in ws)) for ws, x in zip(sw, raw('title') + ' ' + raw('text'))]
    for c in ['query', 'title', 'text']:
        df[c] = df[c].map(clean)
    df['char_count'] = df.article_char_count.map(num)
    df['src'] = df.ingest_source.str[1].astype(int)

    ql = df['query'].map(lemmas); tl = df.title.map(lemmas); xl = df.text.map(lemmas)
    F = pd.DataFrame(index=df.index)
    F['q_len'] = ql.map(len); F['t_len'] = xl.map(len); F['title_len'] = tl.map(len)
    F['title_empty'] = (df.title == '').astype(int)
    for name, L in [('txt', xl), ('ttl', tl)]:
        sets = L.map(set)
        inter = [len(set(q) & s) for q, s in zip(ql, sets)]
        F[f'ov_{name}'] = inter
        F[f'ovr_{name}'] = np.array(inter) / F.q_len.clip(lower=1)
    F['ovr_both'] = [len(set(q) & (set(a) | set(b))) / max(len(q), 1) for q, a, b in zip(ql, xl, tl)]

    # BM25 on corpus-wide idf (all unique passages), scored per row
    docs = df.drop_duplicates('document_id')
    dl = (tl + xl)[docs.index]
    bm = BM25Okapi(dl.tolist())
    pos = {d: i for i, d in enumerate(docs.document_id)}
    # score only the doc's own row: compute manually via bm internals
    F['bm25'] = [_bm25_one(bm, q, pos[d]) for q, d in zip(ql, df.document_id)]
    bmt = BM25Okapi([l if l else ['_'] for l in tl[docs.index]])
    F['bm25_title'] = [_bm25_one(bmt, q, pos[d]) for q, d in zip(ql, df.document_id)]

    # tf-idf cosines (lemma words, char n-grams)
    lj = lambda L: L.map(' '.join)
    for name, a, b, kw in [('tfw', lj(ql), lj(tl + xl), dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                           ('tfc', df['query'].str.lower(), (df.title + ' ' + df.text).str.lower(),
                            dict(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True, max_features=300000))]:
        v = TfidfVectorizer(**kw).fit(pd.concat([a, b.drop_duplicates()]))
        A = v.transform(a); B = v.transform(b)
        F[name] = np.asarray(A.multiply(B).sum(1)).ravel()

    F['q_stress'], F['t_stress'], F['stress_hit'] = q_stress.astype(int), t_stress.astype(int), stress_hit
    # metadata
    F['pidx'] = df.passage_index; F['pcnt'] = df.article_passage_count
    F['prel'] = df.passage_index / df.article_passage_count
    F['chars'] = df.char_count; F['src'] = df.src
    F['txt_chars'] = df.text.str.len()
    F['n_art_in_q'] = df.groupby(['query_id', 'article_id']).id.transform('size')
    F['n_q_doc'] = df.groupby('document_id').query_id.transform('nunique')
    F['n_q_art'] = df.groupby('article_id').query_id.transform('nunique')
    F['q_size'] = df.groupby('query_id').id.transform('size')
    # in-query relative versions of key scores
    for c in ['bm25', 'bm25_title', 'tfw', 'tfc', 'ovr_txt', 'ovr_ttl', 'ovr_both']:
        g = F.groupby(df.query_id)[c]
        F[c + '_rk'] = g.rank(pct=True)
        F[c + '_dmax'] = F[c] - g.transform('max')
    # article-level max of bm25 within query (other passages of the same article)
    F['bm25_art_max'] = F.groupby([df.query_id, df.article_id]).bm25.transform('max')
    F['tfw_art_max'] = F.groupby([df.query_id, df.article_id]).tfw.transform('max')

    meta = df[['id', 'query_id', 'document_id', 'article_id', 'relevance']].reset_index(drop=True)
    out = pd.concat([meta, F.reset_index(drop=True)], axis=1)
    out['is_test'] = out.relevance.isna()
    out.to_parquet('data/feats.parquet')
    df[['id', 'query', 'title', 'text']].to_parquet('data/clean.parquet')
    print(out.shape)


def _bm25_one(bm, q, i):
    s, f, dl = 0.0, bm.doc_freqs[i], bm.doc_len[i]
    for w in q:
        tf = f.get(w, 0)
        if tf:
            s += bm.idf.get(w, 0) * tf * (bm.k1 + 1) / (tf + bm.k1 * (1 - bm.b + bm.b * dl / bm.avgdl))
    return s


if __name__ == '__main__':
    assert clean('&lt;p&gt;Навигация: x\n\nА\xa0 б</p>') == 'А б', clean('&lt;p&gt;Навигация: x\n\nА\xa0 б</p>')
    assert num('3 646,0') == 3646.0 and num('22 311') == 22311
    assert clean('Десяти́нная це́рковь') == 'Десятинная церковь'
    assert clean('Cписок ATФ eё Boeing Microlife') == 'Список АТФ её Boeing Microlife'
    build()


Writing feats.py


In [4]:
%%writefile ce.py
"""Neural scorers: bi-encoder cosine, cross-encoder zero-shot / fine-tuned (OOF)."""
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'


def pair_text(title, text):
    return f'{title}. {text}' if title else text


QWEN_PRE = ('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. '
            'Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
            '<Instruct>: Given a Russian search query, retrieve relevant passages that answer the query\n<Query>: ')
QWEN_SUF = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


GEMMA_PROMPT = ("Given a query A and a passage B, determine whether the passage contains an answer to the query "
                "by providing a prediction of either 'Yes' or 'No'.")


def lm_template(name, tok):
    """(query prefix, doc prefix, suffix) of each LLM reranker's own prompt format."""
    if 'gemma' in name.lower():  # BAAI/bge-reranker-v2-gemma (FlagEmbedding format)
        return tok.bos_token + 'A: ', '\nB: ', '\n' + GEMMA_PROMPT
    return QWEN_PRE, '\n<Document>: ', QWEN_SUF


def fmt(name, queries, docs, max_len=512):
    """LLM rerankers expect their own prompt; the doc body is token-capped so truncation never cuts the suffix."""
    if not any(k in name.lower() for k in ('qwen', 'gemma')):
        return list(queries), list(docs)
    tok = AutoTokenizer.from_pretrained(name)
    pre, mid, suf = lm_template(name, tok)
    qs = [pre + q for q in queries]
    ql = [len(t) for t in tok(qs, add_special_tokens=False)['input_ids']]
    head, sl = len(tok(mid, add_special_tokens=False)['input_ids']), len(tok(suf, add_special_tokens=False)['input_ids'])
    body = tok(list(docs), add_special_tokens=False)['input_ids']
    ds = [f'{mid}{tok.decode(b[:max(max_len - 8 - n - head - sl, 16)])}{suf}' for b, n in zip(body, ql)]
    return qs, ds


def _xlmr_compat():
    """jina-reranker-v2's own (remote) code imports a helper that transformers 5 removed; restore it unchanged."""
    import transformers.models.xlm_roberta.modeling_xlm_roberta as X
    if not hasattr(X, 'create_position_ids_from_input_ids'):
        def create_position_ids_from_input_ids(input_ids, padding_idx, past_key_values_length=0):
            mask = input_ids.ne(padding_idx).int()
            return ((torch.cumsum(mask, dim=1).type_as(mask) + past_key_values_length) * mask).long() + padding_idx
        X.create_position_ids_from_input_ids = create_position_ids_from_input_ids


def load_ce(name):
    remote = 'jina' in name.lower()  # jinaai/jina-reranker-v2-base-multilingual ships its own model code
    if remote:
        _xlmr_compat()
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=remote)
    if tok.pad_token is None:  # decoder rerankers (Qwen3): score sits on the last non-pad token
        tok.pad_token = tok.eos_token
    m = AutoModelForSequenceClassification.from_pretrained(name, num_labels=1, trust_remote_code=remote).to(DEV)
    m.config.pad_token_id = tok.pad_token_id
    return tok, m


@torch.no_grad()
def score_pairs(model, queries, docs, bs=64, max_len=512):
    tok, m = model
    m.eval()
    order = np.argsort([len(d) for d in docs])  # length-sorted batches: less padding
    out = np.zeros(len(docs), dtype=np.float32)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([queries[j] for j in idx], [docs[j] for j in idx], truncation='only_second',
                  max_length=max_len, padding=True, return_tensors='pt').to(DEV)
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == 'cuda'):
            out[idx] = m(**enc).logits.float().view(-1).cpu().numpy()
    return out


def make_batches(lens, bs, seed, pool=50):
    """Shuffle, sort by length inside pools of bs*pool, cut into batches, shuffle batches."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(lens))
    out = []
    for p in range(0, len(idx), bs * pool):
        chunk = sorted(idx[p:p + bs * pool], key=lambda i: lens[i])
        out += [chunk[i:i + bs] for i in range(0, len(chunk), bs)]
    rng.shuffle(out)
    return out


def query_batches(qid, y, seed, qpb=2, max_docs=12):
    """Listwise batches: qpb queries per step, each query's rows together (<= max_docs: all positives + sampled negatives)."""
    rng = np.random.default_rng(seed)
    groups = []
    for rows in pd.Series(np.arange(len(qid))).groupby(np.asarray(qid)).apply(list):
        rows = np.array(rows)
        if len(rows) > max_docs:
            pos, neg = rows[y[rows] > 0], rows[y[rows] == 0]
            rows = np.concatenate([pos, rng.choice(neg, max(max_docs - len(pos), 0), replace=False)])
        groups.append(list(rows))
    order = rng.permutation(len(groups))
    return [[groups[g] for g in order[i:i + qpb]] for i in range(0, len(order), qpb)]


def train_fold(name, train_df, fold, epochs=1, lr=2e-5, bs=16, micro=8, max_len=512, seed=0, loss='bce', teacher=None, T=2.0):
    """Fine-tune cross-encoder with BCE on rows where fold != `fold`. train_df: query, doc, relevance, fold.
    Effective batch bs, split into micro-batches (no gradient checkpointing).
    loss='list': each step = 2 whole queries, BCE + softmax cross-entropy over the query's candidates.
    teacher (scores aligned with train_df rows, e.g. zero-shot Qwen-8B): + cross-entropy to softmax(teacher / T) over the query."""
    torch.manual_seed(seed)
    tok, m = load_ce(name)
    big = loss == 'list' or 'qwen' in name.lower()
    if big:  # T4 15 GB: whole-query forwards / 0.6B decoder only fit with checkpointing
        m.config.use_cache = False
        m.gradient_checkpointing_enable()
    if 'qwen' in name.lower():  # 151k-token embedding: freeze it (saves ~2 GB of Adam state)
        m.get_input_embeddings().weight.requires_grad_(False)
        micro = 4
    d = train_df[train_df.fold != fold]
    q, x, y = d['query'].tolist(), d['doc'].tolist(), d.relevance.to_numpy(np.float32)
    tea = None if teacher is None else np.asarray(teacher, np.float32)[(train_df.fold != fold).to_numpy()]
    assert tea is None or np.isfinite(tea).all(), 'teacher score missing for a training row'
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    batches = lambda ep: (query_batches(d.query_id.to_numpy(), y, seed + ep) if loss == 'list'
                          else [[b[j:j + micro] for j in range(0, len(b), micro)]
                                for b in make_batches([len(t) for t in x], bs, seed + ep)])
    steps = epochs * len(batches(0))
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=DEV == 'cuda')
    lossf = torch.nn.BCEWithLogitsLoss(reduction='sum')
    m.train()
    for ep in range(epochs):
        for s, b in enumerate(batches(ep)):
            opt.zero_grad()
            n = sum(len(mb) for mb in b)
            for mb in b:
                enc = tok([q[i] for i in mb], [x[i] for i in mb], truncation='only_second', max_length=max_len,
                          padding=True, return_tensors='pt').to(DEV)
                with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == 'cuda'):
                    logit = m(**enc).logits.view(-1).float()
                t = torch.tensor(y[mb], device=DEV)
                l = lossf(logit, t) / n
                if loss == 'list':
                    l = l - (t / t.sum().clamp(min=1) * torch.log_softmax(logit, 0)).sum() / len(b)
                if tea is not None:
                    soft = torch.softmax(torch.tensor(tea[mb], device=DEV) / T, 0)
                    l = l - (soft * torch.log_softmax(logit, 0)).sum() / len(b)
                scaler.scale(l).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            if s % 200 == 0:
                print(f'fold {fold} ep {ep} step {s}/{steps // epochs}', flush=True)
    return tok, m


def oof_and_test(fold_scores, df, col='ce_ft'):
    """fold_scores: {fold: scores for all rows of df}. df has is_test, fold. Train row in fold k <- model k; test <- mean."""
    s = np.full(len(df), np.nan)
    for k, sc in fold_scores.items():
        mask = (~df.is_test.to_numpy()) & (df.fold.to_numpy() == k)
        s[mask] = sc[mask]
    te = df.is_test.to_numpy()
    s[te] = np.mean([sc[te] for sc in fold_scores.values()], axis=0)
    return pd.DataFrame({'id': df.id.to_numpy(), col: s})


@torch.no_grad()
def score_lm(name, queries, docs, bs=8):
    """Zero-shot LLM reranker, last position, left padding. Qwen3: logit('yes') - logit('no'); bge-gemma: logit('Yes').
    Models too big for one T4 in fp16 (8B) are split across all visible GPUs."""
    from transformers import AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(name, padding_side='left')
    split = '8b' in name.lower() and torch.cuda.device_count() > 1
    m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, **({'device_map': 'auto'} if split else {}))
    m = (m if split else m.to(DEV)).eval()
    gemma = 'gemma' in name.lower()
    yes = tok('Yes', add_special_tokens=False)['input_ids'][0] if gemma else tok.convert_tokens_to_ids('yes')
    no = tok.convert_tokens_to_ids('no')
    qs, ds = fmt(name, queries, docs)
    texts = [a + b for a, b in zip(qs, ds)]
    order = np.argsort([len(t) for t in texts])
    out = np.zeros(len(texts), dtype=np.float32)
    dev = m.get_input_embeddings().weight.device
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([texts[j] for j in idx], padding=True, return_tensors='pt', add_special_tokens=False).to(dev)
        lg = m(**enc, logits_to_keep=1).logits[:, -1].float()
        out[idx] = (lg[:, yes] if gemma else lg[:, yes] - lg[:, no]).cpu().numpy()
    return out


def with_position(d):
    """Prefix the passage with its place in the article, e.g. 'Фрагмент 1 из 24.' (skipped where metadata is missing)."""
    pos = [f'Фрагмент {int(i) + 1} из {int(n)}. ' if i == i and n == n else (f'Фрагмент {int(i) + 1}. ' if i == i else '')
           for i, n in zip(d.passage_index, d.article_passage_count)]
    return [p + x for p, x in zip(pos, d.doc)]


def load_pairs(code_dir, train_csv):
    d = pd.read_parquet(f'{code_dir}/clean.parquet').merge(pd.read_csv(f'{code_dir}/folds.csv'), on='id', how='left')
    d = d.merge(pd.read_csv(train_csv, usecols=['id', 'relevance']), on='id', how='left')
    ids = pd.concat([pd.read_csv(f, usecols=['id', 'query_id', 'passage_index', 'article_passage_count'])
                     for f in (train_csv, train_csv.replace('train.csv', 'test.csv'))])
    d = d.merge(ids, on='id', how='left')
    d['is_test'] = d.relevance.isna()
    d['doc'] = [pair_text(t, x) for t, x in zip(d.title, d.text)]
    return d


if __name__ == '__main__':  # worker: python ce.py <code_dir> <train_csv> <fold> <model> <out_dir> [seed] [loss] [tag]
    import sys
    code_dir, train_csv, fold, name, out = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4], sys.argv[5]
    seed = int(sys.argv[6]) if len(sys.argv) > 6 else 0
    loss = sys.argv[7] if len(sys.argv) > 7 else 'bce'  # bce | list | lm (zero-shot LLM reranker); suffixes +pos, +ns
    tag = sys.argv[8] if len(sys.argv) > 8 else 'ce'
    d = load_pairs(code_dir, train_csv)
    if loss.endswith('+ns'):  # strip stress marks (U+0301) from query and passage
        d['query'], d['doc'], loss = d['query'].str.replace('\u0301', ''), d.doc.str.replace('\u0301', ''), loss[:-3]
    teacher = None
    if loss.endswith('+kd'):  # distil the zero-shot Qwen-8B scores (computed earlier in the same notebook)
        d = d.merge(pd.read_parquet(f'{code_dir}/nn_q8.parquet'), on='id', how='left')
        teacher, loss = d.ce_q8.to_numpy(), loss[:-3]
    if loss.endswith('+pos'):
        d['doc'], loss = with_position(d), loss[:-4]
    if loss == 'lm':  # no training; fold < 0 -> score every row
        import hashlib  # fold k: train rows of fold k + a fixed 1/5 of test rows -> 5 jobs cover everything once
        part = d.id.map(lambda i: int(hashlib.md5(i.encode()).hexdigest(), 16) % 5)
        need = d if fold < 0 else d[(~d.is_test & (d.fold == fold)) | (d.is_test & (part == fold))]
        sc = score_lm(name, need['query'].tolist(), need.doc.tolist())
    else:
        d['query'], d['doc'] = fmt(name, d['query'], d.doc)
        tr = ~d.is_test.to_numpy()
        m = train_fold(name, d[tr], fold, seed=seed, loss=loss, teacher=None if teacher is None else teacher[tr])
        m[1].half()
        need = d[d.is_test | (d.fold == fold)]  # only held-out fold + test are ever used
        sc = score_pairs(m, need['query'].tolist(), need.doc.tolist())
    pd.DataFrame({'id': need.id, 'score': sc}).to_parquet(f'{out}/{tag}_fold{fold}_s{seed}.parquet')
    print('fold done', fold, 'seed', seed, flush=True)


Writing ce.py


In [5]:
%%writefile cv.py
"""Group CV of level-2 rankers on data/feats.parquet (+ data/nn_*.parquet). Run: python cv.py"""
import glob, hashlib, os
import numpy as np, pandas as pd, lightgbm as lgb

META = ('id', 'query_id', 'document_id', 'article_id', 'relevance', 'is_test', 'fold')
DISC = 1 / np.log2(np.arange(2, 12))


def ndcg10(df, col):
    """Kaggle metric: sort by score desc, tie -> id asc; IDCG over all candidates."""
    res = []
    for _, g in df.sort_values('id').groupby('query_id'):
        rel = g.relevance.to_numpy()[np.argsort(-g[col].to_numpy(), kind='stable')]
        k = min(10, len(rel))
        dcg = ((2 ** rel[:k] - 1) * DISC[:k]).sum()
        idcg = ((2 ** np.sort(rel)[::-1][:k] - 1) * DISC[:k]).sum()
        res.append(dcg / idcg if idcg else 0)
    return float(np.mean(res))


def make_folds(train):
    """fold = md5(query_id) % 5: identical on any machine / library version."""
    fold = train.query_id.map(lambda q: int(hashlib.md5(q.encode()).hexdigest(), 16) % 5)
    return pd.DataFrame({'id': train.id.to_numpy(), 'fold': fold.to_numpy()})


CTX = ('ce_ft', 'ce_li', 'ce_q4', 'ce_q8', 'ce_zs', 'ce_jn', 'bi_cos')


def add_ctx(d):
    """Neural scores relative to the same article's other passages and to the whole query."""
    g, q = d.groupby(['query_id', 'article_id']), d.groupby('query_id')
    for c in [c for c in CTX if c in d]:
        d[c] = d[c].astype('float64')
        d[c + '_amax'] = g[c].transform('max'); d[c + '_arank'] = g[c].rank(ascending=False)
        d[c + '_qz'] = (d[c] - q[c].transform('mean')) / (q[c].transform('std') + 1e-6)
    return d


def load():
    d = pd.read_parquet('data/feats.parquet')
    for p in sorted(glob.glob('data/nn_*.parquet')):
        d = d.merge(pd.read_parquet(p), on='id', how='left')
    if not os.path.exists('data/folds.csv'):
        make_folds(d[~d.is_test]).to_csv('data/folds.csv', index=False)
    d = d.merge(pd.read_csv('data/folds.csv'), on='id', how='left')
    return add_ctx(d.sort_values(['query_id', 'id']).reset_index(drop=True))


LGB = dict(objective='lambdarank', metric='ndcg', eval_at=[10], learning_rate=0.02, num_leaves=15,
           min_data_in_leaf=40, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, verbose=-1)


def fit_lgb(tr, te, feats, rounds=600, seeds=(0, 1, 2)):
    """LightGBM lambdarank per fold; seeds averaged by in-query rank."""
    oof, pred = np.zeros(len(tr)), np.zeros(len(te))
    for s in seeds:
        o, p = np.zeros(len(tr)), np.zeros(len(te))
        for f in range(5):
            A, B = tr[tr.fold != f], tr[tr.fold == f]
            m = lgb.train({**LGB, 'seed': s},
                          lgb.Dataset(A[feats], A.relevance, group=A.groupby('query_id', sort=False).size().to_numpy()),
                          num_boost_round=rounds)
            o[B.index] = m.predict(B[feats]); p += m.predict(te[feats]) / 5
        oof += qrank(tr, o); pred += qrank(te, p) + 1e-6 * p
    return oof, pred


def fit_cat(tr, te, feats, iters=1500):
    from catboost import CatBoostRanker, Pool
    oof, pred = np.zeros(len(tr)), np.zeros(len(te))
    for f in range(5):
        A, B = tr[tr.fold != f], tr[tr.fold == f]
        m = CatBoostRanker(loss_function='YetiRank', iterations=iters, learning_rate=0.05, depth=6,
                           random_seed=0, verbose=0, thread_count=-1)
        m.fit(Pool(A[feats], A.relevance, group_id=A.query_id))
        oof[B.index] = m.predict(B[feats]); pred += m.predict(te[feats]) / 5
    return oof, pred


def qrank(df, s):
    """Per-query percentile rank, so models on different scales can be averaged."""
    return pd.Series(s, index=df.index).groupby(df.query_id).rank(pct=True).to_numpy()


def main(submit=False):
    d = load()
    tr, te = d[~d.is_test].reset_index(drop=True), d[d.is_test].reset_index(drop=True)
    feats = [c for c in d.columns if c not in META]
    for c in [c for c in ['bm25', 'tfc', 'bi_cos', 'ce_zs', 'ce_jn', 'ce_q4', 'ce_q8', 'ce_ft', 'ce_li'] if c in d]:
        print(f'{c:8s} {ndcg10(tr, c):.4f}')
    res = {'lgb': fit_lgb(tr, te, feats), 'cat': fit_cat(tr, te, feats)}
    for k, (o, _) in res.items():
        tr[k] = o
        print(f'{k} OOF {ndcg10(tr, k):.4f}  folds {np.round([ndcg10(tr[tr.fold == f], k) for f in range(5)], 4)}')
    tr['blend'] = sum(qrank(tr, o) for o, _ in res.values())
    print(f'blend OOF {ndcg10(tr, "blend"):.4f}  folds {np.round([ndcg10(tr[tr.fold == f], "blend") for f in range(5)], 4)}')
    if submit:
        # rank-average + tiny raw-score term to break ties inside a query
        p = sum(qrank(te, t) for _, t in res.values()) + 1e-6 * sum(t for _, t in res.values())
        pd.DataFrame({'id': te.id, 'prediction': p}).to_csv('submission.csv', index=False)
        print('wrote submission.csv')


if __name__ == '__main__':
    import sys; main(submit="--submit" in sys.argv)


Writing cv.py


## 1. Чистка и лексические признаки, фолды

In [6]:
!python feats.py
import cv
_ = cv.load()  # creates data/folds.csv

(45037, 48)


## 2. Zero-shot нейро-признаки

In [7]:
import numpy as np, pandas as pd, torch, ce
from sentence_transformers import SentenceTransformer
MODEL = 'BAAI/bge-reranker-v2-m3'
d = ce.load_pairs('data', 'data/train.csv')
qs, docs = d['query'].tolist(), d.doc.tolist()
bi = SentenceTransformer('deepvk/USER-bge-m3', device='cuda'); bi.half()
uq = sorted(set(qs)); qe = dict(zip(uq, bi.encode(uq, batch_size=128, normalize_embeddings=True)))
de = bi.encode(docs, batch_size=64, normalize_embeddings=True)
del bi; torch.cuda.empty_cache()
m = ce.load_ce(MODEL); m[1].half()
zs = ce.score_pairs(m, qs, docs)
del m; torch.cuda.empty_cache()
j = ce.load_ce('jinaai/jina-reranker-v2-base-multilingual'); j[1].half()  # zero-shot, own model code
pd.DataFrame({'id': d.id, 'bi_cos': [qe[q] @ e for q, e in zip(qs, de)], 'ce_zs': zs,
              'ce_jn': ce.score_pairs(j, qs, docs)}).to_parquet('data/nn_zs.parquet')
del j; torch.cuda.empty_cache()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.34k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.44GB            

... (36 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.


Loading weights:   0%|          | 0/153 [00:00<?, ?it/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


## 3. Реранкеры: дообучение bge (5 фолдов на конфигурацию) и zero-shot Qwen3-Reranker-8B (на обоих GPU); задачи раздаются свободным GPU

In [8]:
import queue, subprocess, sys, threading
# (tag -> output column, model, loss, seeds)
RERANKERS = [('li', 'BAAI/bge-reranker-v2-m3', 'list', [0, 1]), ('q8', 'Qwen/Qwen3-Reranker-8B', 'lm', [0]), ('ce', 'BAAI/bge-reranker-v2-m3', 'bce', [0])]
jobs = queue.Queue()
fails = []
# 8B rerankers do not fit one T4 in fp16: run them first, each job on all GPUs
for tag, name, loss, seeds in [r for r in RERANKERS if '8B' in r[1]]:
    for f in range(5):
        if subprocess.run([sys.executable, 'ce.py', 'data', 'data/train.csv', str(f), name, 'data', '0', loss, tag]).returncode:
            fails.append((tag, 0, f))
for tag, name, loss, seeds in [r for r in RERANKERS if '8B' not in r[1]]:
    for s in seeds:
        for f in range(5): jobs.put((tag, name, loss, s, f))
def gpu_worker(g):
    while True:
        try: tag, name, loss, s, f = jobs.get_nowait()
        except queue.Empty: return
        r = subprocess.run([sys.executable, 'ce.py', 'data', 'data/train.csv', str(f), name, 'data', str(s), loss, tag],
                           env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)})
        if r.returncode: fails.append((tag, s, f))
ts = [threading.Thread(target=gpu_worker, args=(g,)) for g in range(max(torch.cuda.device_count(), 1))]
[t.start() for t in ts]; [t.join() for t in ts]
assert not fails, fails
# average seeds per fold, then OOF for train / mean of folds for test
rd = lambda tag, f, s: pd.read_parquet(f'data/{tag}_fold{f}_s{s}.parquet').set_index('id').score.reindex(d.id).to_numpy()
for tag, name, loss, seeds in RERANKERS:
    col = 'ce_ft' if tag == 'ce' else f'ce_{tag}'
    if loss == 'lm':  # zero-shot: the 5 jobs are disjoint slices of all rows
        x = pd.concat([pd.read_parquet(f'data/{tag}_fold{f}_s0.parquet') for f in range(5)])
        assert x.id.is_unique and set(x.id) == set(d.id)
        x.rename(columns={'score': col}).to_parquet(f'data/nn_{tag}.parquet'); continue
    sc = {f: np.mean([rd(tag, f, s) for s in seeds], axis=0) for f in range(5)}
    ce.oof_and_test(sc, d, col).to_parquet(f'data/nn_{tag}.parquet')

Loading weights: 100%|██████████| 399/399 [00:37<00:00, 10.68it/s]


fold done 0 seed 0


Loading weights: 100%|██████████| 399/399 [00:39<00:00, 10.17it/s]


fold done 1 seed 0


Loading weights: 100%|██████████| 399/399 [00:12<00:00, 31.07it/s]


fold done 2 seed 0


... (30 outputs left out)

/kaggle/working/ce.py:152: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step()


fold 3 ep 0 step 200/1577
fold done 2 seed 0


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2697.20it/s]
/kaggle/working/ce.py:152: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step()


fold 4 ep 0 step 0/1550
fold 3 ep 0 step 400/1577
fold 4 ep 0 step 200/1550
fold 3 ep 0 step 600/1577
fold 4 ep 0 step 400/1550
fold 4 ep 0 step 600/1550
fold 3 ep 0 step 800/1577
fold 4 ep 0 step 800/1550
fold 3 ep 0 step 1000/1577
fold 4 ep 0 step 1000/1550
fold 3 ep 0 step 1200/1577
fold 4 ep 0 step 1200/1550
fold 3 ep 0 step 1400/1577
fold 4 ep 0 step 1400/1550
fold done 3 seed 0
fold done 4 seed 0


## 4. Ранжировщики второго уровня и сабмит

In [9]:
cv.main(submit=True)
import shutil; shutil.copy('submission.csv', '/kaggle/working/submission.csv') if os.path.exists('/kaggle/working') and os.getcwd() != '/kaggle/working' else None
pd.read_csv('submission.csv').describe()

bm25     0.7105
tfc      0.7109
bi_cos   0.8922
ce_zs    0.9215
ce_jn    0.9536
ce_q8    0.9397
ce_ft    0.9352
ce_li    0.9404
lgb OOF 0.9661  folds [0.9591 0.9657 0.9675 0.9726 0.9657]
cat OOF 0.9644  folds [0.9574 0.9645 0.9653 0.9686 0.9662]
blend OOF 0.9659  folds [0.9591 0.9651 0.9677 0.9709 0.9665]
wrote submission.csv


,prediction
count,13507.000000
mean,1.096618
std,0.568739
min,0.117642
25%,0.599998
50%,1.090911
75%,1.600005
max,2.000015


In [10]:
import importlib.metadata as im
for p in ['numpy', 'pandas', 'scikit-learn', 'pyarrow', 'lightgbm', 'catboost', 'torch', 'transformers',
          'sentence-transformers', 'huggingface-hub', 'pymorphy3', 'pymorphy3-dicts-ru', 'rank-bm25']:
    print(f'{p}=={im.version(p)}')

numpy==2.1.3
pandas==2.3.3
scikit-learn==1.6.1
pyarrow==23.0.1
lightgbm==4.6.0
catboost==1.2.10
torch==2.11.0+cu130
transformers==5.20.2
sentence-transformers==5.7.0
huggingface-hub==1.33.0
pymorphy3==2.0.6
pymorphy3-dicts-ru==2.4.417150.4580142
rank-bm25==0.2.2
